# Notebook 01: Exploração e Validação das Camadas Bronze
**Projeto Integrador: Da Ingestão à Decisão (ENEM + ANEEL)**

Este notebook serve para prototipação rápida, inspeção das amostras baixadas via API REST e CSV, e validação dos 4 metadados técnicos obrigatórios da Camada Bronze:
- `_ingestion_time` (Timestamp UTC da ingestão)
- `_source` (Origem oficial dos dados)
- `_load_id` (Identificador único UUID da carga)
- `_record_hash` (Hash SHA-256 para garantia de idempotência e deduplicação)

In [ ]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd()
bronze_dir = base_dir / 'data' / 'bronze'
if not bronze_dir.exists():
    bronze_dir = base_dir.parent / 'data' / 'bronze'

print(f'Diretório Bronze localizado em: {bronze_dir.resolve()}')

## 1. Inspeção dos Dados Brutos da ANEEL (API REST CKAN)

In [ ]:
mapa_file = bronze_dir / 'aneel' / 'aneel_conjuntos_municipios.parquet'
df_mapa = pd.read_parquet(mapa_file)
print(f'Mapeamento Conjunto x Município (PA): {len(df_mapa)} registros')
df_mapa.head(5)

In [ ]:
ind_files = list((bronze_dir / 'aneel').glob('aneel_dec_fec_*.parquet'))
df_ind = pd.concat([pd.read_parquet(f) for f in ind_files], ignore_index=True)
print(f'Indicadores DEC/FEC brutos ingeridos: {len(df_ind):,} linhas')
print(f'Distribuição de indicadores:\n{df_ind["SigIndicador"].value_counts().head(8)}')
df_ind.head(5)

## 2. Inspeção dos Microdados Brutos do ENEM (CSV/Parquet)

In [ ]:
enem_files = list((bronze_dir / 'enem').glob('enem_*.parquet'))
df_enem = pd.concat([pd.read_parquet(f) for f in enem_files], ignore_index=True)
print(f'Total de microdados de candidatos na Bronze: {len(df_enem):,} registros')
print(f'Anos encontrados: {sorted(df_enem["NU_ANO"].dropna().unique().tolist())}')
df_enem.head(5)

## 3. Validação dos Metadados Técnicos Obrigatórios

In [ ]:
metadados = ['_ingestion_time', '_source', '_load_id', '_record_hash']
for col in metadados:
    assert col in df_ind.columns, f'Metadado {col} ausente na ANEEL!'
    assert col in df_enem.columns, f'Metadado {col} ausente no ENEM!'
print('Auditoria de Metadados: [OK] Todos os 4 metadados técnicos presentes e válidos.')